<a href="https://colab.research.google.com/github/humptybigdump/inprogress/blob/main/04_2_ISE_2025_NLP_N_gram_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**To adapt this notebook to your own needs** and to be able to edit it, please make a copy of your own. This works via "*File*" -> "*Save a copy ..*."


---



In Sect. 2.9 *Language Models* of the **ISE 2025 lecture** we also introduce the concept of statistical **language models**, in particular the **n-gram model**.  


# N-gram Model

A **statistical language model** is a probability distribution over sequences of words. Given such a sequence, it assigns a probability to the whole sequence.  For our next examples, again, we chose the "Gutenberg" corpus.

In [ ]:
#First we have to import nltk and download a few required packages
import nltk
#We choose the gutenberg corpus ...
nltk.download('gutenberg')

#...and have a look, which texts are contained in it
nltk.corpus.gutenberg.fileids()

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


['austen-emma.txt',
 'austen-persuasion.txt',
 'austen-sense.txt',
 'bible-kjv.txt',
 'blake-poems.txt',
 'bryant-stories.txt',
 'burgess-busterbrown.txt',
 'carroll-alice.txt',
 'chesterton-ball.txt',
 'chesterton-brown.txt',
 'chesterton-thursday.txt',
 'edgeworth-parents.txt',
 'melville-moby_dick.txt',
 'milton-paradise.txt',
 'shakespeare-caesar.txt',
 'shakespeare-hamlet.txt',
 'shakespeare-macbeth.txt',
 'whitman-leaves.txt']

Again, we use a sample text, e.g. **Shakespeare's Julius Caesar** (`shakespeare-caesar.txt`).

In [ ]:
#We shortcut to gutenberg to keep sequences shorter
from nltk.corpus import gutenberg

caesar = gutenberg.words('shakespeare-caesar.txt') #creates a list of words from the text
print(caesar[:20])

['[', 'The', 'Tragedie', 'of', 'Julius', 'Caesar', 'by', 'William', 'Shakespeare', '1599', ']', 'Actus', 'Primus', '.', 'Scoena', 'Prima', '.', 'Enter', 'Flauius', ',']


nltk offers us the possibility to tokenise a word sequence directly into **n-grams.**

In [ ]:
from nltk import bigrams #here we demonstrate bigrams
nltk.download('punkt_tab') #the "punkt_tab" package is required for tokenization

first_sentence = gutenberg.sents('shakespeare-caesar.txt')[5] #our first example are the first five sentences of Julius Caesar
print(first_sentence)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['Hence', ':', 'home', 'you', 'idle', 'Creatures', ',', 'get', 'you', 'home', ':', 'Is', 'this', 'a', 'Holiday', '?']


First, let's create **bigrams**:

In [ ]:
from collections import defaultdict #we need the defaultdictionary library for counting bigrams in the box below

print(list(bigrams(first_sentence, pad_left=True, pad_right=True)))

[(None, 'Hence'), ('Hence', ':'), (':', 'home'), ('home', 'you'), ('you', 'idle'), ('idle', 'Creatures'), ('Creatures', ','), (',', 'get'), ('get', 'you'), ('you', 'home'), ('home', ':'), (':', 'Is'), ('Is', 'this'), ('this', 'a'), ('a', 'Holiday'), ('Holiday', '?'), ('?', None)]


We **count the bigrams** in the word sequence:

In [ ]:
model = defaultdict(lambda: defaultdict(lambda: 0)) #our language model will be implemented in a python dictionary of dictionaries

for sentence in gutenberg.sents('shakespeare-caesar.txt'): #for all sentences
    #words should be lowercase (lower()), remove all punctuation (isalpha())
    words = [word.lower() for word in sentence if word.isalpha()]

    for w1, w2 in bigrams(words, pad_right=True, pad_left=True): #now we count bigrams according to their occurrence
        model[w1][w2] += 1

# 2 example bigram counts
print(model["you"]["idle"])
print(model[None]["the"]) #How often is "the" the beginning of a sentence

1
22


We already have the single bigram counts. Now we can easily compute the **bigram probability** via **maximum likelihood estimation** (cf. ISE lecture).

In [ ]:
#bigram probabilities according to the formula in the ISE lecture 05, slide 04
for w1 in model:
    total_w1 = float(sum(model[w1].values())) #wn-1
    for w2 in model[w1]:
        model[w1][w2] /= total_w1  #wn-1 wn / #wn-1

# print two example bigram probabilities
print(model["you"]["idle"])
print(model[None]["the"])

0.0025575447570332483
0.010171058714748035


# Example from ISE lecture 5

Now let's try out the **bigram** probability example from the lecture. We've had given the following corpus:

`I saw the boy. The man is working. I walked in the street.`

In the lecture, we have added special tokens to indicate the beginning and the end of the sentences:

`<s>I saw the boy<\s>. <s>The man is working.<\s> <s>I walked in the street.<\s>`

**However**, as we have pointed out in the lecture, we do not necessarily need two separate tokens  `<s>, <\s>` for computing a language model. Therefore, we simply use **ONE SINGLE TOKEN** and for further simplification, **we use "." (period)** instead of `<s>`.

In [ ]:
from nltk import word_tokenize #import word tokenizer

model = defaultdict(lambda: defaultdict(lambda: 0)) #this is the data structure for the bigram probability table

text = "I saw the boy. The man is working. I walked in the street."
words = word_tokenize(text.lower()) #For simplification, we use lower case. Here, we don't necessarily need this.

#print the example text as bigrams. Note: we complement a period "." in the beginning of the first sentence, since "." indicates the start/end of a sentence
print(list(bigrams(words, pad_left=True, left_pad_symbol='.')))

#now we count all occurrences of bigrams and put them in the model
for w1, w2 in bigrams(words, pad_left=True, left_pad_symbol='.'):
    model[w1][w2] += 1

#how often does the bigram ". I" occur?
print(model["."]["i"])

#computing bigram probabilities by dividing bigram counts by unigram count of the preceding word #wn-1 wn / #wn-1
for w1 in model:
    total_w1 = float(sum(model[w1].values()))
    for w2 in model[w1]:
        model[w1][w2] /= total_w1

#what is the probability of bigram ". I"?
print(model["."]["i"])

[('.', 'i'), ('i', 'saw'), ('saw', 'the'), ('the', 'boy'), ('boy', '.'), ('.', 'the'), ('the', 'man'), ('man', 'is'), ('is', 'working'), ('working', '.'), ('.', 'i'), ('i', 'walked'), ('walked', 'in'), ('in', 'the'), ('the', 'street'), ('street', '.')]
2
0.6666666666666666


In [ ]:
#Now determine the probability of the following sendence: "I saw the man"
text2 = "I saw the man"
words2 = word_tokenize(text2.lower())
print(list(bigrams(words2, pad_left=True, left_pad_symbol='.'))) #print the bigram list

text2bigram = bigrams(words2, pad_left=True, left_pad_symbol='.') #preparing the text as bigrams for further processing

#simply multiply all probabilities of text2 bigrams according to the already computed language model
prob = 1.0
for bi in text2bigram:
    prob *= model[bi[0]][bi[1]]

print(prob)

[('.', 'i'), ('i', 'saw'), ('saw', 'the'), ('the', 'man')]
0.1111111111111111


More information on how to use corpora with NLTK:


*   Steven Bird, Ewan Klein, and Edward Loper: [Natural Language Processing with Python
– Analyzing Text with the Natural Language Toolkit](https://www.nltk.org/book/), O'Reilly Media, 2009
> * Chap 2: [Accessing Text Corpora and Lexical Resources](https://www.nltk.org/book/ch02.html)
*   Generating Donald Trump Tweets with [N-Gram Language Models with NLTK](https://www.kaggle.com/code/alvations/n-gram-language-model-with-nltk) at kaggle



